In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
BASELINE = "00_00_visit"
ANTIBIOTIC_COLUMN = "oralheatlh_antibiotics_for_gingivitis"
ANTIBIOTIC_USE = 1


def clean_ids(s):
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


diet = pd.read_csv(DATA_DIR / "csv_diet" / "diet_logging_events_cleaned_baseline.csv", low_memory=False)
oral = pd.read_csv(DATA_DIR / "csv_oral" / "oral_microbiome_data.csv", low_memory=False)
oral_health = pd.read_csv(DATA_DIR / "covariates" / "oral_health.csv", low_memory=False)
for df in (diet, oral, oral_health):
    df["participant_id"] = clean_ids(df["participant_id"])

In [ ]:
diet = diet[diet["research_stage"] == BASELINE]
oral = oral[oral["research_stage"] == BASELINE].copy()
oral["oral_collection_date"] = pd.to_datetime(oral["collection_date"], errors="coerce", utc=True).dt.tz_convert(None)
oral = oral.sort_values(["participant_id", "oral_collection_date"]).drop_duplicates("participant_id")

cohort = diet[["participant_id"]].drop_duplicates().merge(
    oral[["participant_id", "research_stage", "oral_collection_date"]], on="participant_id")

oral_health = oral_health[oral_health["research_stage"] == BASELINE]
antibiotic_use = pd.to_numeric(oral_health[ANTIBIOTIC_COLUMN], errors="coerce") == ANTIBIOTIC_USE
n_matched = len(cohort)
cohort = cohort[~cohort["participant_id"].isin(oral_health.loc[antibiotic_use, "participant_id"])]

print("matched at the baseline visit:", n_matched)
print("excluded for recent antibiotic use:", n_matched - len(cohort))
print("final cohort:", len(cohort))

In [ ]:
cohort.to_csv(DATA_DIR / "csv_diet" / "select_diet_oral.csv", index=False)
diet[diet["participant_id"].isin(cohort["participant_id"])].to_csv(
    DATA_DIR / "csv_diet" / "select_diet_oral_baseline_participants_diet_logging.csv", index=False)